# Shortest Path (Wroclaw Public Transport Stops) Problem
- Djikstra's algorithm
- A* algorithm

In [31]:
import pandas as pd
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
import time

from pandas.core.interchange.dataframe_protocol import DataFrame


## Load Data

In [85]:
def load_df(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    return df

raw_df = load_df("data.csv")

In [86]:
print("Data Loaded\n")
print(raw_df.head())

print("\nData Types\n")
print(raw_df.info())

print("\nDescription\n")
print(raw_df.describe())

print("\nMarginal departure_times\n")
print(raw_df["departure_time"].max(), raw_df["departure_time"].min(), "\n")

print("\nMarginal arrival_time\n")
print(raw_df["arrival_time"].max(), raw_df["arrival_time"].min(), "\n")

Data Loaded

   Unnamed: 0       company line departure_time arrival_time  \
0           0  MPK Autobusy    A       20:52:00     20:53:00   
1           1  MPK Autobusy    A       20:53:00     20:54:00   
2           2  MPK Autobusy    A       20:54:00     20:55:00   
3           3  MPK Autobusy    A       20:55:00     20:57:00   
4           4  MPK Autobusy    A       20:57:00     20:59:00   

             start_stop              end_stop  start_stop_lat  start_stop_lon  \
0   Zajezdnia Obornicka              Paprotna       51.148737       17.021069   
1              Paprotna  Obornicka (Wołowska)       51.147752       17.020539   
2  Obornicka (Wołowska)            Bezpieczna       51.144385       17.023735   
3            Bezpieczna              Bałtycka       51.141360       17.026376   
4              Bałtycka         Broniewskiego       51.136632       17.030617   

   end_stop_lat  end_stop_lon  
0     51.147752     17.020539  
1     51.144385     17.023735  
2     51.141360    

## Normalize Data

In [89]:
def fix_time_format(time_str) -> str:
    return f"{int(time_str.split(":")[0]) % 24:02d}:{time_str.split(':')[1]}:{time_str.split(':')[2]}"

def normalize_data(df: pd.DataFrame) -> pd.DataFrame:
    df["line"] = df["line"].str.strip().str.lower()
    df["company"] = df["company"].str.strip().str.lower()

    df["departure_time"] = df["departure_time"].apply(lambda x: fix_time_format(x))
    df["arrival_time"] = df["arrival_time"].apply(lambda x: fix_time_format(x))

    df["departure_time"] = pd.to_datetime(df["departure_time"], format="%H:%M:%S")
    df["arrival_time"] = pd.to_datetime(df["arrival_time"], format="%H:%M:%S")

    df["start_stop"] = df["start_stop"].str.strip().str.lower()
    df["end_stop"] = df["end_stop"].str.strip().str.lower()

    return df

normalized_df = normalize_data(raw_df.copy())
print("\nNormalized Data\n")
print(normalized_df)


Normalized Data

        Unnamed: 0               company line      departure_time  \
0                0          mpk autobusy    a 1900-01-01 20:52:00   
1                1          mpk autobusy    a 1900-01-01 20:53:00   
2                2          mpk autobusy    a 1900-01-01 20:54:00   
3                3          mpk autobusy    a 1900-01-01 20:55:00   
4                4          mpk autobusy    a 1900-01-01 20:57:00   
...            ...                   ...  ...                 ...   
996515      996515  dla kąty wrocławskie  967 1900-01-01 18:38:00   
996516      996516  dla kąty wrocławskie  967 1900-01-01 18:39:00   
996517      996517  dla kąty wrocławskie  967 1900-01-01 18:41:00   
996518      996518  dla kąty wrocławskie  967 1900-01-01 18:42:00   
996519      996519  dla kąty wrocławskie  967 1900-01-01 18:43:00   

              arrival_time                   start_stop  \
0      1900-01-01 20:53:00          zajezdnia obornicka   
1      1900-01-01 20:54:00         

In [90]:
print("\nMarginal departure_times\n")
print(normalized_df["departure_time"].max(), normalized_df["departure_time"].min(), "\n")

print("\nMarginal arrival_time\n")
print(normalized_df["arrival_time"].max(), normalized_df["arrival_time"].min(), "\n")


Marginal departure_times

1900-01-01 23:59:00 1900-01-01 00:00:00 


Marginal arrival_time

1900-01-01 23:59:00 1900-01-01 00:00:00 



In [93]:
def load_graph(df: pd.DataFrame) -> nx.DiGraph:
    graph_ = nx.MultiDiGraph()  # Multi-Edges Directed graph

    for _, row in df.iterrows():
        travel_time = (
            row["arrival_time"] - row["departure_time"]
        ).seconds  # Travel time in seconds

        if travel_time >= 0:
            graph_.add_edge(
                row["start_stop"],
                row["end_stop"],
                # key=f"{row["line"]}-{row["start_stop"]}-{row["departure_time"]}",
                weight=travel_time,
                line=row["line"],
                departure_time=row["departure_time"].time,
                arrival_time=row["arrival_time"].time,
                start_lat=row["start_stop_lat"],
                start_lon=row["start_stop_lon"],
                end_lat=row["end_stop_lat"],
                end_lon=row["end_stop_lon"],
            )

    return graph_

graph = load_graph(normalized_df)
print(graph)

MultiDiGraph with 939 nodes and 996520 edges
